# Propylene Oxide (−) PEELD — 2W, 20260820

Photoelectron **elliptical** dichroism from the `PO (-) PEELD 2W` dataset.

The QWP was stepped through a symmetric set of rotations; each rotation
angle θ produces elliptically polarised light with ellipticity
ε = tan θ (ε = 0 linear, ε = 1 circular). The chosen angles
±14.04°, ±26.57°, ±36.87°, ±45.00° correspond to
ε = 0.25, 0.50, 0.75, 1.00.

For each ellipticity magnitude the two opposite-handedness runs
(+θ vs −θ) are antisymmetrised the same way as the PECD analysis:

$$\mathrm{PECD}(\varepsilon) = 2\left(\frac{F_{+}-B_{+}}{F_{+}+B_{+}}
  - \frac{F_{-}-B_{-}}{F_{-}+B_{-}}\right)$$

`py` is the light-propagation axis: forward = `py > 0`, backward = `py < 0`.
The θ = 0 (linear) run is the control.

In [ ]:
import glob
import os
import re

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

In [ ]:
DATA_DIR = r"J:\ctgroup\Edward\DATA\VMI\20260820\PO (-) PEELD 2W"

PR_MAX = 0.8
MQ_MAX = 65


def ellipticity(qwp_deg):
    """Ellipticity of the light after a QWP rotated by `qwp_deg` from linear."""
    return np.tan(np.deg2rad(qwp_deg))


data = pd.DataFrame()
for path in sorted(glob.glob(os.path.join(DATA_DIR, "*_calibrated.h5"))):
    qwp = float(re.search(r"([+-]?\d+\.\d+)_calibrated\.h5$", os.path.basename(path)).group(1))
    d = pd.read_hdf(path)
    d["qwp"] = qwp
    d["ellipticity"] = ellipticity(qwp)
    data = pd.concat([data, d], ignore_index=True)

data["pr"] = np.sqrt(data["px"] ** 2 + data["py"] ** 2 + data["pz"] ** 2)
data = data[(data["pr"] < PR_MAX) & (data["m/q"] < MQ_MAX)]

# ellipticity rounded to 2 dp: signed for display, magnitude to pair +/- theta
data["eps"] = data["ellipticity"].round(2)
data["ell_mag"] = data["ellipticity"].abs().round(2)

ELL_MAGS = sorted(m for m in data["ell_mag"].unique() if m > 1e-6)
lin = data[data["qwp"] == 0.0]

print(data.groupby(["qwp", "eps"]).size())
print("ellipticity pairs:", ELL_MAGS)
data.head()

In [ ]:
def fba(df):
    """Forward-backward asymmetry (F - B) / (F + B) along py."""
    f = int((df["py"] > 0).sum())
    b = int((df["py"] < 0).sum())
    n = f + b
    if n == 0:
        return np.nan, np.nan, 0
    a = (f - b) / n
    err = np.sqrt((1 - a ** 2) / n)  # binomial error on the asymmetry
    return a, err, n


def pecd_in_bins(dfp, dfm, df0, col, edges, min_counts=1500):
    """Antisymmetrized PECD (+theta vs -theta) and raw FBA per bin of `col`."""
    rows = []
    for lo, hi in zip(edges[:-1], edges[1:]):
        pb = dfp[(dfp[col] > lo) & (dfp[col] <= hi)]
        mb = dfm[(dfm[col] > lo) & (dfm[col] <= hi)]
        zb = df0[(df0[col] > lo) & (df0[col] <= hi)]
        ap, ep, np_ = fba(pb)
        am, em, nm = fba(mb)
        a0, e0, n0 = fba(zb)
        n = np_ + nm
        pecd = 2 * (ap - am) if n >= min_counts else np.nan
        pecd_err = 2 * np.sqrt(ep ** 2 + em ** 2) if n >= min_counts else np.nan
        rows.append(
            dict(
                center=0.5 * (lo + hi),
                pecd=pecd,
                pecd_err=pecd_err,
                fba_plus=ap,
                fba_minus=am,
                fba_lin=a0,
                counts=n,
            )
        )
    return pd.DataFrame(rows)


def pair(ell_mag):
    """(+theta run, -theta run) DataFrames for a given ellipticity magnitude."""
    m = data["ell_mag"] == ell_mag
    return data[m & (data["ellipticity"] > 0)], data[m & (data["ellipticity"] < 0)]


# one colour per ellipticity, reused across every figure
ELL_COLORS = dict(zip(
    ELL_MAGS,
    px.colors.sample_colorscale("Viridis", np.linspace(0.0, 0.85, len(ELL_MAGS))),
))

In [ ]:
# m/q spectrum, to see which fragments carry signal
px.histogram(
    data, x="m/q", color="eps", nbins=400, barmode="overlay", opacity=0.5,
    range_x=[0, MQ_MAX], title="m/q spectrum",
)

In [ ]:
# integrated forward-backward asymmetry per ellipticity
a0, e0, _ = fba(lin)
print(f"FBA(linear) = {a0:+.4f} +/- {e0:.4f}   (control, should be ~0)\n")
for ell in ELL_MAGS:
    sp, sm = pair(ell)
    ap, ep, _ = fba(sp)
    am, emag, _ = fba(sm)
    pecd = 2 * (ap - am)
    perr = 2 * np.sqrt(ep ** 2 + emag ** 2)
    print(
        f"eps = {ell:.2f}:  FBA(+) = {ap:+.4f} +/- {ep:.4f}   "
        f"FBA(-) = {am:+.4f} +/- {emag:.4f}   "
        f"PECD = {pecd:+.4f} +/- {perr:.4f}"
    )

## Forward-backward asymmetry vs radial momentum

In [ ]:
PR_EDGES = np.linspace(0, PR_MAX, 20)

fig = go.Figure()
last = None
for ell in ELL_MAGS:
    sp, sm = pair(ell)
    d = pecd_in_bins(sp, sm, lin, "pr", PR_EDGES, min_counts=1500)
    last = d
    fig.add_trace(go.Scatter(
        x=d["center"], y=d["pecd"],
        error_y=dict(type="data", array=d["pecd_err"], thickness=1),
        mode="lines+markers", line_color=ELL_COLORS[ell],
        name=f"eps = {ell:.2f}",
    ))
fig.add_trace(go.Scatter(
    x=last["center"], y=last["fba_lin"], mode="lines",
    line=dict(color="gray", dash="dash"), name="FBA (linear)",
))
fig.add_hline(y=0, line_color="black", line_width=1)
fig.update_layout(
    title="PEELD vs radial momentum",
    xaxis_title="radial momentum p_r (a.u.)",
    yaxis=dict(title="asymmetry", range=[-0.3, 0.3], zeroline=True),
    height=600, width=850, template="simple_white+presentation",
    legend=dict(x=0.02, y=0.98),
)
fig.show()

## Forward-backward asymmetry vs m/q

In [ ]:
MQ_EDGES = np.linspace(0, MQ_MAX, 30)
mq_centers = 0.5 * (MQ_EDGES[:-1] + MQ_EDGES[1:])
mq_spectrum, _ = np.histogram(data["m/q"], bins=MQ_EDGES)

fig = go.Figure()
fig.add_trace(go.Bar(
    x=mq_centers, y=mq_spectrum, name="m/q spectrum", yaxis="y2",
    marker_color="lightgray", opacity=0.5,
))
for ell in ELL_MAGS:
    sp, sm = pair(ell)
    d = pecd_in_bins(sp, sm, lin, "m/q", MQ_EDGES, min_counts=2000)
    fig.add_trace(go.Scatter(
        x=d["center"], y=d["pecd"],
        error_y=dict(type="data", array=d["pecd_err"], thickness=1),
        mode="lines+markers", line_color=ELL_COLORS[ell],
        name=f"eps = {ell:.2f}",
    ))
fig.add_hline(y=0, line_color="black", line_width=1)
fig.update_layout(
    title="PEELD vs m/q",
    xaxis=dict(title="m/q", range=[0, MQ_MAX]),
    yaxis=dict(title="PECD", range=[-0.2, 0.2], zeroline=True),
    yaxis2=dict(title="counts", overlaying="y", side="right", showgrid=False),
    height=600, width=850, template="simple_white+presentation",
    legend=dict(x=0.02, y=0.98),
)
fig.show()

## px–pz intensity maps per fragment (|p_y| < 0.05)

The four strongest fragments in the m/q spectrum. Rows are the fragments
(plus a combined "total"), and the nine columns are the individual QWP
runs — both handedness signs shown separately — labelled by
ellipticity. Each panel is the px–pz photoelectron distribution in the
central slice `|py| < 0.05 a.u.`

In [ ]:
PY_SLICE = 0.05      # keep |py| < PY_SLICE  (a.u.)
MQ_BIN_W = 0.25      # m/q histogram bin width for peak finding
PEAK_FRAC = 0.05     # a bin counts as signal if > PEAK_FRAC * tallest bin
N_PEAKS = 4          # keep the N strongest fragments
GRID_BINS = 120      # px/pz heatmap resolution

# --- fragments = continuously connected m/q bins above threshold ---
mq_edges = np.arange(0, MQ_MAX + MQ_BIN_W, MQ_BIN_W)
mq_counts, _ = np.histogram(data["m/q"], bins=mq_edges)
above = mq_counts > PEAK_FRAC * mq_counts.max()

fragments = []  # (lo, hi, centroid, n_ions)
i = 0
while i < len(above):
    if not above[i]:
        i += 1
        continue
    j = i
    while j < len(above) and above[j]:
        j += 1
    lo, hi = mq_edges[i], mq_edges[j]
    m = data["m/q"].between(lo, hi)
    fragments.append((lo, hi, data.loc[m, "m/q"].mean(), int(m.sum())))
    i = j

fragments = sorted(sorted(fragments, key=lambda f: -f[3])[:N_PEAKS],
                   key=lambda f: f[2])
print("top fragments:", [(round(c, 1), n) for _, _, c, n in fragments])

edges = np.linspace(-PR_MAX, PR_MAX, GRID_BINS + 1)
centers = 0.5 * (edges[:-1] + edges[1:])
sl = data[data["py"].abs() < PY_SLICE]


def img(df):
    h, _, _ = np.histogram2d(df["px"], df["pz"], bins=[edges, edges])
    return h.T


# rows: total + one per fragment; columns: one per ellipticity
all_frag = np.zeros(len(sl), dtype=bool)
row_masks = []
for lo, hi, c, n in fragments:
    s = sl["m/q"].between(lo, hi).to_numpy()
    all_frag |= s
    row_masks.append((f"m/q {c:.1f}", s))
rows = [("total", all_frag)] + row_masks

# one column per QWP run, ordered by (signed) ellipticity
runs = (data[["qwp", "eps"]].drop_duplicates().sort_values("eps")
        .itertuples(index=False))
runs = list(runs)

fig = make_subplots(
    rows=len(rows), cols=len(runs),
    row_titles=[r[0] for r in rows],
    column_titles=[f"eps = {run.eps:+.2f}" for run in runs],
    horizontal_spacing=0.012, vertical_spacing=0.03,
)
k = 0
for r, (_, mq_mask) in enumerate(rows):
    for cc, run in enumerate(runs):
        sub = sl[mq_mask & (sl["qwp"] == run.qwp).to_numpy()]
        fig.add_trace(
            go.Heatmap(z=img(sub), x=centers, y=centers,
                       colorscale="Inferno", showscale=False),
            row=r + 1, col=cc + 1,
        )
        ax = "" if k == 0 else str(k + 1)
        fig.update_yaxes(scaleanchor=f"x{ax}", constrain="domain", row=r + 1, col=cc + 1)
        fig.update_xaxes(showticklabels=False, row=r + 1, col=cc + 1)
        fig.update_yaxes(showticklabels=False, row=r + 1, col=cc + 1)
        k += 1

fig.update_layout(
    title=f"px-pz intensity by QWP run, |py| < {PY_SLICE} a.u.",
    height=230 * len(rows), width=170 * len(runs),
    template="simple_white",
)
fig.show()